# Simulation times

Randomly sampled 100 parameter vectors per problem for general stats and selection of the best sensitivity method.


## Load data

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

from lib import PROBLEM_OVERVIEW_PATH

problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)

In [ ]:
# load / prepare data
rows = []
for problem_id in problem_df.id:
    with open(f"../data/simulation_times/{problem_id}.json") as f:
        d = json.load(f)
    print(problem_id)
    rows.extend(d)

df = pd.DataFrame(rows)
del rows

df["obj_success"] = np.isfinite(df.fval)
df = df.join(
    problem_df.rename(columns={"short": "problem_short"}).set_index("id")[
        ["n_conditions", "problem_short", "problem_color", "amici_nx_solver"]
    ],
    on="problem_id",
)
df.sensitivity_method = pd.Categorical(
    df.sensitivity_method.map({"none": "none", "fsa": "FSA", "asa": "ASA"}),
    categories=("none", "FSA", "ASA"),
    ordered=True,
)
df_imploded = df

df = (
    df.explode(
        [
            "amici_status",
            "amici_cpu_times_total_s",
            "condition_ids",
            "amici_messages",
        ]
    )
    .rename(
        columns={
            "amici_cpu_times_total_s": "amici_cpu_time_total_s",
            "condition_ids": "condition_id",
        }
    )
    .reset_index(drop=True)
)
df["condition_success"] = df.amici_status == "AMICI_SUCCESS"
df["amici_cpu_time_total_s"] = df["amici_cpu_time_total_s"].astype(float)

df.head()

In [ ]:
df.amici_status.value_counts()

In [ ]:
# same number of replicates everywhere
assert (
    1
    == df.groupby(["problem_id", "sample_idx"])
    .count()
    .reset_index()[["problem_id", "sample_idx"]]
    .groupby("problem_id")
    .count()
    .sample_idx.nunique()
)

assert np.all((df.amici_cpu_time_total_s > 0) | (~df.condition_success))

## Stats

In [ ]:
t_tot = df_imploded.time_obj_s.sum()
print(
    f"Total CPU time was about {t_tot / 60:.0f} minutes ({t_tot / 60 / 60:.2f} hours)"
)

In [ ]:
df_tmp = df.query(
    "sensitivity_method == 'none' and condition_success"
).amici_cpu_time_total_s
print(
    f"Simulation time per condition: {df_tmp.min() * 1000:.3g}--{df_tmp.max() * 1000:.3g}ms"
)

df_tmp = df_imploded.query(
    "sensitivity_method == 'none' and obj_success"
).time_obj_s
print(
    f"Time for objective evaluation: {df_tmp.min() * 1000:.3g}--{df_tmp.max() * 1000:.4g}ms"
)

In [ ]:
# Average
df_imploded.groupby("sensitivity_method").agg({"obj_success": "mean"})

 ## No sensitivities (Figure S2)


In [ ]:
# 1 row corresponds to one simulation of one condition
df

In [ ]:
from matplotlib.collections import PolyCollection

with plt.rc_context(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial"],
        "font.size": 7,
        "lines.markersize": 3,
        "savefig.bbox": None,
    }
):
    fig = plt.figure(figsize=(18 / 2.54, 8), dpi=300)
    gs = fig.add_gridspec(
        3,
        2,
        width_ratios=[1, 1],
        height_ratios=[1, 1, 1.5],
        hspace=1.1,
        left=0.05,
        right=0.95,
        top=0.9,
    )

    ax_top = fig.add_subplot(gs[0, :])
    ax_center = fig.add_subplot(gs[1, :])
    ax_bottom_left = fig.add_subplot(gs[2, 0])
    ax_bottom_right = fig.add_subplot(gs[2, 1])
    subfig_font = {
        "fontsize": 9,
        "fontweight": "bold",
    }

    ax = ax_top

    fig.text(0.0, 0.95, "A", va="top", ha="left", **subfig_font)
    fig.text(
        0.05,
        0.95,
        "Computation time for numerical simulation of an experimental condition",
        va="top",
        ha="left",
        **subfig_font,
    )

    df_tmp = df.query("sensitivity_method == 'none' and condition_success")
    df_tmp.amici_cpu_time_total_s = np.log10(df_tmp.amici_cpu_time_total_s)

    ax = sns.violinplot(
        df_tmp,
        x="problem_short",
        y="amici_cpu_time_total_s",
        inner=None,
        cut=0,
        ax=ax,
        density_norm="width",
        order=problem_df.short.sort_values().values,
    )
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("log10(Computation time [s])")
    ax.set_xlabel("Problem")

    polys = [c for c in ax.collections if isinstance(c, PolyCollection)]
    colors = problem_df.set_index("short").sort_index().problem_color.values
    for poly, color in zip(polys, colors, strict=True):
        poly.set_facecolor(color)

    fig.text(0.0, 0.65, "B", va="top", ha="left", **subfig_font)
    fig.text(
        0.05,
        0.65,
        "Computation time for objective function evaluation",
        ha="left",
        va="top",
        **subfig_font,
    )

    # note that for the failed ones, many conditions were potentially skipped
    df_tmp = df_imploded.query("sensitivity_method == 'none' and obj_success")
    df_tmp.time_obj_s = np.log10(df_tmp.time_obj_s)

    ax = ax_center
    sns.violinplot(
        df_tmp,
        x="problem_short",
        y="time_obj_s",
        inner=None,
        cut=0,
        ax=ax,
        density_norm="width",
        order=problem_df.short.sort_values().values,
    )
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("log10(Computation time [s])")
    ax.set_xlabel("Problem")
    polys = [c for c in ax.collections if isinstance(c, PolyCollection)]
    colors = problem_df.set_index("short").sort_index().problem_color.values
    for poly, color in zip(polys, colors, strict=True):
        poly.set_facecolor(color)

    fig.text(0.0, 0.35, "C", va="top", ha="left", **subfig_font)
    fig.text(
        0.05,
        0.35,
        "Correlation of computation time\nand number of state variables",
        ha="left",
        va="top",
        **subfig_font,
    )

    ax = ax_bottom_left
    ax.set_xscale("log")
    ax.set_yscale("log")
    data = (
        df_imploded.query("sensitivity_method == 'none' and obj_success")
        .groupby(["problem_short", "amici_nx_solver", "problem_color"])[
            "time_obj_s"
        ]
        .mean()
        .reset_index(["amici_nx_solver", "problem_color"])
    )
    ax.scatter(
        data.amici_nx_solver,
        data.time_obj_s,
        color=data.problem_color,
    )
    ax.set_xlabel("Number of state variables")
    ax.set_ylabel("Computation time [s]")
    x = np.log10(data.amici_nx_solver)
    y = np.log10(data.time_obj_s)
    corr_res = spearmanr(x, y)
    ax.text(
        0.05,
        0.95,
        f"$\\rho_s$ = {corr_res.correlation:.2f}\np = {corr_res.pvalue:.2e}",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    fig.text(0.5, 0.35, "D", va="top", ha="left", **subfig_font)
    fig.text(
        0.55,
        0.35,
        "Correlation of computation time\nand number of experimental conditions",
        ha="left",
        va="top",
        **subfig_font,
    )

    ax = ax_bottom_right
    ax.set_xscale("log")
    ax.set_yscale("log")
    data = (
        df_imploded.query("sensitivity_method == 'none' and obj_success")
        .groupby(["problem_short", "n_conditions", "problem_color"])[
            "time_obj_s"
        ]
        .mean()
        .reset_index(["n_conditions", "problem_color"])
    )
    ax.scatter(
        data.n_conditions,
        data.time_obj_s,
        color=data.problem_color,
    )
    ax.set_xlabel("Number of conditions")
    ax.set_ylabel("Computation time [s]")
    x = np.log10(data.n_conditions)
    y = np.log10(data.time_obj_s)
    corr_res = spearmanr(x, y)
    ax.text(
        0.05,
        0.95,
        f"$\\rho_s$ = {corr_res.correlation:.2f}\np = {corr_res.pvalue:.2e}",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    fig.align_labels()

    plt.savefig("out/FigureS2.pdf", bbox_inches="tight")

## With sensitivities (method selection) (Figure S3)


In [ ]:
with plt.rc_context(
    {
        "font.size": 8,
        "figure.dpi": 300,
        "lines.linewidth": 0.8,
        "lines.markersize": 4,
    },
):
    fig = plt.figure(figsize=(18 / 2.54, 9), layout="constrained")
    gs = fig.add_gridspec(4, 2)  # 4 rows, 2 cols

    # three full-width rows, shared x
    axs = [fig.add_subplot(gs[i, :]) for i in range(3)]
    for ax in axs[1:]:
        ax.sharex(axs[0])
    for ax in axs[:-1]:
        ax.label_outer()

    # Time for objective evaluation, FSA vs ASA
    df_tmp = df_imploded.query("sensitivity_method != 'none' and obj_success")
    df_tmp["sensitivity_method"] = df_tmp[
        "sensitivity_method"
    ].cat.remove_unused_categories()
    df_tmp["log10_time_obj_s"] = np.log10(df_tmp["time_obj_s"])

    ax = axs[0]
    sns.violinplot(
        df_tmp.rename(columns={"sensitivity_method": "Sensitivity method"}),
        x="problem_short",
        y="log10_time_obj_s",
        hue="Sensitivity method",
        inner=None,
        cut=0,
        ax=ax,
        density_norm="width",
    )
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel("Time for gradient evaluation\nlog10(Computation time [s])")
    ax.set_xlabel("Problem")
    ax.legend(loc="upper right")

    # failure rate
    df_tmp = df_imploded.query("sensitivity_method != 'none'")
    df_tmp["sensitivity_method"] = df_tmp[
        "sensitivity_method"
    ].cat.remove_unused_categories()
    success_rate_df = df_tmp.groupby(
        ["problem_short", "sensitivity_method"]
    ).agg(
        success_rate=pd.NamedAgg("obj_success", "mean"),
        mean_time_s=pd.NamedAgg("time_obj_s", "mean"),
        min_time_s=pd.NamedAgg("time_obj_s", "min"),
        max_time_s=pd.NamedAgg("time_obj_s", "max"),
    )
    # mean_of_all / success_rate
    success_rate_df["t_eff"] = (
        success_rate_df.mean_time_s / success_rate_df.success_rate
    )

    ax = axs[1]
    ax = sns.barplot(
        success_rate_df.reset_index(),
        x="problem_short",
        y="success_rate",
        hue="sensitivity_method",
        order=sorted(df.problem_short.unique()),
        ax=ax,
    )
    ax.tick_params(axis="x", rotation=90)
    ax.legend().remove()
    ax.set_ylabel("Success rate\nof gradient evaluation")

    ax = axs[2]
    order = sorted(df.problem_short.unique())
    ax = sns.barplot(
        success_rate_df.reset_index(),
        x="problem_short",
        y="t_eff",
        hue="sensitivity_method",
        order=order,
        ax=ax,
    )
    ax.legend().remove()

    # Salazar: FSA always failed => t_eff undefined
    point_in_axes = np.array([order.index("SalazarCavazos") - 0.25, 0])
    point_in_data = ax.transAxes.inverted().transform(
        ax.transData.transform(point_in_axes)
    )
    ax.text(
        point_in_data[0],
        0,
        "NA",
        ha="center",
        va="bottom",
        transform=ax.transAxes,
    )

    ax.tick_params(axis="x", rotation=90)
    ax.set_yscale("log")
    ax.set_ylabel("Effective time\nfor gradient evaluation [s]")
    ax.set_xlabel("Problem")

    for ax in axs:
        for i in range(df_tmp.problem_short.nunique()):
            ax.axvline(i + 0.5, c="gray", linestyle=":", alpha=0.2)

    ax = fig.add_subplot(gs[3, 0])
    df_tmp = (
        df_imploded.query("sensitivity_method != 'none' and obj_success")
        .groupby(["problem_id", "sensitivity_method"])["time_obj_s"]
        .median()
        .reset_index()
    )
    df_tmp = df_tmp.join(
        problem_df.rename(columns={"short": "problem_short"}).set_index("id")[
            ["n_est_parameters"]
        ],
        on="problem_id",
    )

    df_tmp["sensitivity_method"] = df_tmp[
        "sensitivity_method"
    ].cat.remove_unused_categories()
    ax = sns.scatterplot(
        df_tmp,
        x="n_est_parameters",
        y="time_obj_s",
        hue="sensitivity_method",
        ax=ax,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("# Estimated parameters")
    ax.set_ylabel("Time for gradient evaluation [s]")
    ax.set_title("D", loc="left", x=-0.3, fontweight="bold", fontsize=10)
    ax.get_legend().remove()

    for ax, title in zip(axs, "ABC", strict=False):
        ax.set_title(
            title, loc="left", x=-0.15, fontweight="bold", fontsize=10
        )

    fig.align_ylabels()

    plt.savefig("out/FigureS3.pdf", bbox_inches="tight")
    plt.savefig("out/FigureS3.svg", bbox_inches="tight")

In [ ]:
success_rate_df